# makemore — activations, gradients & BatchNorm

Cleaned-up notes from Karpathy's *Zero to Hero* (makemore part 3). Same MLP as before; the focus now is **making it train well**: sane initialization, healthy activations and gradients, and batch normalization.

## Setup and data

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(words[:8])
print(len(words))

# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

## Fixing initialization — two problems at step 0

A freshly-initialized net has two avoidable pathologies, both visible in the very first loss:

1. **Over-confident output layer.** If `W2`/`b2` are large, the initial logits are extreme, so the softmax is wildly (and wrongly) confident and the first loss is huge. Scaling `W2` down (×0.01) and setting `b2 ≈ 0` makes the initial distribution near-uniform, so training starts from a sensible loss instead of wasting steps just squashing the logits.

2. **Saturated tanh.** If the pre-activations are too broad, `tanh` outputs pile up at ±1, where its gradient is ~0 — those neurons barely learn. Scaling `W1` by `(5/3)/sqrt(fan_in)` keeps the pre-activations in a good range (details two cells down).

In [ ]:
# MLP revisited
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5) #* 0.2. # this solves thetanh problem below just a fancy less hacky method.
b1 = torch.randn(n_hidden,                        generator=g) * 0.01 # this solves thetanh problem below
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01 # we want them to be low value weights but cannot be 0 thats bad so a small values
b2 = torch.randn(vocab_size,                      generator=g) * 0.   # we want all close to 0 here all zero else the randomeness will just make the first loss ver high

# BatchNorm parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

In [ ]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
for i in range(max_steps) :
    # minibatch construct
    ix = torch. randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

    # forward pass
    emb = C[Xb] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    hpreact = embcat @ W1 + b1 # hidden layer pre-activation
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
        p.grad = None

    loss. backward()

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad
    # track stats
    if i % 10000 == 0: # print every once in a while
        print (f'{i: 7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss. log10().item())

## Diagnosing the tanh saturation

White = a neuron whose output is > 0.99 in absolute value, i.e. saturated. Too much white means many neurons are stuck at the flat tails of `tanh`, where gradients vanish and learning stalls. (sigmoid/ReLU have their own versions of this; leaky-ReLU, maxout, ELU largely avoid it.)

In [ ]:
# plt.hist(h.view(-1).tolist(), 50)
# here we see the second problem we see most values are set to 1 or -1 this is due to the tanh smoothing problem we see
# the values of hpreact is very broad from the line below 
# plt.hist(hpreact.view(-1).tolist(), 50)
# this causes the gradients during back propogation to be destroyed (set to 0 or 1) this can cause
plt.figure(figsize=(5,100))
plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest')
# this a problem with sigmoid or relu. leaky relu, maxout and elu do not have this problem

## Why scale by 1/√(fan-in) — Kaiming initialization

Multiplying by a random weight matrix *widens* the distribution of activations (bigger std). To keep the spread roughly constant layer to layer, divide the weights by `sqrt(fan_in)` (the number of inputs). A per-activation gain corrects for the squashing of the non-linearity — `5/3` for `tanh`. This is **Kaiming init**; PyTorch's `torch.nn.init` has the gains for each activation.

In [ ]:
x = torch.randn(1000,10)
w = torch.randn(10,200)
y = x @ w
print(x.mean(), x.std())
print(y.mean(), y.std())
plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.hist(x.view(-1).tolist(), 50, density=True);
plt.subplot (122)
plt.hist(y.view(-1).tolist(), 50, density=True) ;
# thios shows the multiplcation broadens the gaussian distribution ( look at x axis) now we can multiple like we did 0.2 above 
# we do this by dividing by sqrt of the fanin (new term means the number of input columns) (n_embd * block_size in the work) 
# (5/3)/((n_embd * block_size)**0.5) the 5/3 is fix for tanh for other functions https://docs.pytorch.org/docs/2.13/nn.init.html 

## Loss curve and evaluation

In [ ]:
plt.plot(lossi)

In [ ]:

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 # + b1
  #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

## Sampling

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

## Batch normalization

The initialization fixes above are fiddly and don't scale to deep nets. **BatchNorm** sidesteps them: it normalizes each layer's pre-activations to zero-mean, unit-variance *across the batch*, then re-scales and re-shifts with two learned parameters `gamma` (`bngain`) and `beta` (`bnbias`). Placed right after a linear (or conv) layer, it keeps activations healthy regardless of how the weights were initialized.

A few consequences visible in the code:
- **The bias `b1` is dropped.** BatchNorm subtracts the batch mean, which cancels any additive bias — so `b1` does nothing and is removed. `beta` is the effective bias instead.
- **Running mean/std (`momentum`).** At test time you may have a single example and no batch to normalize over. So during training we keep an exponential moving average of the batch mean/std (`0.999 * old + 0.001 * new`; the `0.001` is the momentum) and use those fixed values for inference. (Alternatively, calibrate once over the whole training set at the end — the next cell does this.)
- **Regularization side-effect (your "explain this better" note).** Each example is normalized using the *batch's* statistics, so its activations wobble slightly depending on which other examples happen to land in the same batch. That coupling injects a small, random jitter into every example — a kind of data augmentation on the activations — which makes it harder for the net to memorize and so acts as a mild regularizer. It's a bit of a wart (predictions become batch-dependent), but the regularizing effect is why people tolerate it.

In [ ]:
# MLP revisited
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5) #* 0.2
#b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.randn(vocab_size,                      generator=g) * 0

# BatchNorm parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

In [ ]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
# we have removed the bias b1 in our method as we are adding that value and then by. -bmean we aere removing it so b1 is not doing anything
# so we remove it. We place batchnormalization after multiplcation layers like lieanr layer or convulutional layer
# this batchnormaliaziton helps use control the actiavation of neurons it makes it easri to not focus too much on the initial 
# weights and bias we use  during out training.
for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hpreact = embcat @ W1 #+ b1 # hidden layer pre-activation
  # BatchNorm layer
  # problem of batchnormalization is that in forward pass a value of one input is depenedt on the values they have seen before 
  # this causes jitter but we use it becuase this jitter works like a regularizaation ( by moving creates and type of augmentation)
  #  to prevent overfitting (Explian this better claude)
  # -------------------------------------------------------------
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani # this 0.001 is just the weight need to put for the latest value aslo 
    # known as momentum. 
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
  # -------------------------------------------------------------
  # Non-linearity
  h = torch.tanh(hpreact) # hidden layer
  logits = h @ W2 + b2 # output layer
  loss = F.cross_entropy(logits, Yb) # loss function
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

In [ ]:
# by using batchnormalization we have a problem that now our model expects a 
# calibrate the batch norm at the end of training if we want to predict only one input its not possible as 
# our code exzpects a batch so we use this. this caclaulates the batch mean and standard deviation for the whole
#  training set once and then we use it for validation and testing. another way is during trainign keep track of mean running anf 
# mean std they are updated without any grad chnge the if condition we see in the code. 

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 # + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)

# print(bnmean, bnstd)
# print(bnmean_running, bnstd_running)

In [ ]:

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 # + b1
  #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
#Let's train a deeper network
# The classes we create here are the same API as nn.Module in PyTorch

class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]
# layers = [
#   Linear(n_embd * block_size, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size),
# ]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 5/3 #5/3 # if we use 

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

## Torchifying: `Linear` / `BatchNorm1d` / `Tanh`

Wrap each operation in a small class with the same `__call__` / `parameters()` API as `torch.nn.Module`, so a deep net is just a list of layers. `BatchNorm1d` carries the learned `gamma`/`beta` plus the running-mean/var buffers; note the last layer's `gamma *= 0.1` to start un-confident, and the `5/3` gain on the others.

In [ ]:
# same optimization as last time
max_steps = 20000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  # if i >= 1000:
  #   break # AFTER_DEBUG: would take out obviously to run full optimization


## Diagnostics: activations, gradients, and the update:data ratio

Four health checks for a deep net at initialization/early training:
- **activation distribution** per layer — should be similar across layers, not collapsing to 0 or saturating at ±1;
- **gradient distribution** per layer — likewise should be stable, not shrinking (vanishing) or exploding with depth;
- **weight-gradient distribution** — the grad:data ratio per weight tensor;
- **update:data ratio** (`(lr*grad).std() / data.std()`, log10) — a rule of thumb is roughly **1e-3**; far below means learning too slowly, far above means steps too large.

In [ ]:
# this shows just after one trainign ( wuth the break in to uncommonetd ) how many neurons is problems caused by tanh function
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

In [ ]:

# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

In [ ]:

# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');

In [ ]:

plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);